In [4]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../")

import numpy as np
from src.predictors.chronos import Chronos
from transformers import PreTrainedModel


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/home/skowronek/.conda/envs/mt_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/skowronek/.conda/envs/mt_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-18 08:25:39,918 - INFO - config.py - PyTorch version 2.5.1 available.


# get number of trainable params for last layer fine tuning

In [5]:
def compute_trainable_params(model: PreTrainedModel):
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    fraction_trainable_params = trainable_params / total_params

    return np.round(fraction_trainable_params*100, 2)

In [ ]:
for size in ["tiny", "mini", "small", "base"]:
    chronos = Chronos(**{
                "pretrained_model_name_or_path": f"amazon/chronos-bolt-{size}",
                "device_map": "cuda",
            })
    
    # Freeze all parameters
    for param in chronos.pipeline.inner_model.parameters():
        param.requires_grad = False

    for param in chronos.pipeline.inner_model.output_patch_embedding.output_layer.parameters():
        param.requires_grad = True

    trainable_params = compute_trainable_params(chronos.pipeline.inner_model)

    print(size, trainable_params)

2025-06-17 14:35:37,340 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-tiny
2025-06-17 14:35:37,341 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-tiny
2025-06-17 14:35:38,002 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-mini
2025-06-17 14:35:38,003 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-mini


tiny 6.82


2025-06-17 14:35:39,696 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-small
2025-06-17 14:35:39,697 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-small


mini 4.17


2025-06-17 14:35:42,304 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-base
2025-06-17 14:35:42,305 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-base


small 2.47
base 0.86


In [ ]:
for size in ["tiny", "small", "base", "large"]:
    chronos = Chronos(**{
                "pretrained_model_name_or_path": f"amazon/chronos-t5-{size}",
                "device_map": "cuda",
            })
    
    # Freeze all parameters
    for param in chronos.pipeline.inner_model.parameters():
        param.requires_grad = False

    for param in chronos.pipeline.inner_model.lm_head.parameters():
        param.requires_grad = True

    trainable_params = compute_trainable_params(chronos.pipeline.inner_model)

    print(size, trainable_params)

2025-06-17 14:34:15,958 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-tiny
2025-06-17 14:34:15,959 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-tiny
2025-06-17 14:34:16,504 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-17 14:34:16,509 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-small
2025-06-17 14:34:16,510 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-small


tiny 12.49


2025-06-17 14:34:17,066 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-17 14:34:17,071 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-base
2025-06-17 14:34:17,072 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-base


small 4.54


2025-06-17 14:34:17,870 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-17 14:34:17,876 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-large
2025-06-17 14:34:17,877 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-large


base 1.56


2025-06-17 14:34:44,830 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512


large 0.59


# get number of trainable params for lora fine tuning

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
for size in ["tiny", "small", "base", "large"]:
    chronos = Chronos(**{
                "pretrained_model_name_or_path": f"amazon/chronos-t5-{size}",
                "device_map": "cuda",
            })


    lora_config = LoraConfig(
                    r=8,
                    lora_alpha=8,
                    target_modules=["q", "v", "k"],
                    lora_dropout=0.0,
                    task_type=None,
                )
    
    lora_model = get_peft_model(chronos.pipeline.inner_model, lora_config)
    print("Model:", size)
    lora_model.print_trainable_parameters()

2025-06-17 16:04:55,201 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-tiny
2025-06-17 16:04:55,203 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-tiny


2025-06-17 16:04:55,792 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-17 16:04:55,834 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-small
2025-06-17 16:04:55,834 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-small


Model: tiny
trainable params: 147,456 || all params: 8,541,952 || trainable%: 1.7263


2025-06-17 16:04:56,413 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-17 16:04:56,474 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-base
2025-06-17 16:04:56,475 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-base


Model: small
trainable params: 442,368 || all params: 46,596,608 || trainable%: 0.9494


2025-06-17 16:04:57,345 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512
2025-06-17 16:04:57,460 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-t5-large
2025-06-17 16:04:57,461 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-t5-large


Model: base
trainable params: 1,327,104 || all params: 202,702,080 || trainable%: 0.6547


2025-06-17 16:04:59,023 - INFO - chronos.py - Context length detected of: 2048. Adapt context length to maximum of 512


Model: large
trainable params: 3,538,944 || all params: 712,502,272 || trainable%: 0.4967


In [ ]:
for size in ["tiny", "mini", "small", "base"]:
    chronos = Chronos(**{
                "pretrained_model_name_or_path": f"amazon/chronos-bolt-{size}",
                "device_map": "cuda",
            })


    lora_config = LoraConfig(
                    r=8,
                    lora_alpha=8,
                    target_modules=["q", "v", "k"],
                    lora_dropout=0.0,
                    bias="none",
                    task_type=TaskType.SEQ_2_SEQ_LM,
                )
    lora_model = get_peft_model(chronos.pipeline.inner_model, lora_config)
    print("Model:", size)
    lora_model.print_trainable_parameters()

2025-06-17 15:02:57,308 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-tiny
2025-06-17 15:02:57,309 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-tiny
2025-06-17 15:02:57,894 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-mini
2025-06-17 15:02:57,895 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-mini


Model: tiny
trainable params: 147,456 || all params: 8,800,128 || trainable%: 1.6756


2025-06-17 15:02:58,526 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-small
2025-06-17 15:02:58,526 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-small


Model: mini
trainable params: 258,048 || all params: 21,494,144 || trainable%: 1.2006


2025-06-17 15:02:59,154 - INFO - chronos.py - Loading Chronos pipeline from model: amazon/chronos-bolt-base
2025-06-17 15:02:59,155 - INFO - chronos.py - Initializing Chronos pipeline with model: amazon/chronos-bolt-base


Model: small
trainable params: 442,368 || all params: 48,160,384 || trainable%: 0.9185
Model: base
trainable params: 1,327,104 || all params: 206,620,032 || trainable%: 0.6423
